# Fine-Tuning the CLIP Model on a Dataset of Fashion Product Images

<PRE>
Source:
    https://marqo.ai/course/fine-tuning-clip-models
Adapted:
    Antonio Esteves @ UMinho, May 2025
</PRE>

---

In this notebook, we will look at how we can fine-tune a CLIP model for image classification.

## 1. Install Relevant Libraries

We first install relevant libraries:
```bash
!pip install openai-clip
!pip install datasets
!pip install torch
!pip install tqdm
```

We will use `openai-clip` to define the base CLIP model and we use `datasets` provided by Hugging Face to obtain the dataset. The library `torch` will be used to facilitate model loading, device management, tensor manipulation, and inference. Finally, `tqdm` is used to track the progress of the fine-tuning.

The next step is obtaining a dataset to perform the fine-tuning.

## 2. Load a Dataset

To perform fine-tuning, we will use a small image classification dataset, [fashion-products-small](https://huggingface.co/datasets/ceyda/fashion-products-small), which is a dataset containing 4270 images of 45 categories of fashion products.

In [ ]:
from datasets import load_dataset

# Load the dataset
ds = load_dataset('ceyda/fashion-products-small')

Let us take a look at the features available in the dataset.

In [ ]:
ds

So, for each image there are `filename`, `link`, `id`, `masterCategory`, `gender`, `subCategory` and `image`. Let us print the first sample from the dataset to see what these features mean.

In [ ]:
entry = ds['train'][0]
print(entry)

Thus, the features of the dataset are as follows:

* `filename`: this is the filename of the image.
* `link`: this is a URL to the location of the image file, which is hosted online. This link can be used to view or download the image.
* `id`: this is a unique identifier for the image, which can be used to reference this specific item within the dataset.
* `masterCategory`: this indicates the broad category under which this product falls.
* `gender`: this specifies the intended gender for the product, in this case, men's clothing.
* `subCategory`: this is a more specific category within the master category. "Topwear" indicates that the product is an item of clothing worn on the upper body, such as a shirt, t-shirt, or jacket.
* `image`: this is a PIL (Python Imaging Library) image object, which allows for image manipulation and processing. It specifies the image mode (RGB, meaning it has red, green, and blue color channels) and the image size (384 pixels wide by 512 pixels height).

We will now inspect this image.

In [ ]:
image = entry['image']
image

As expected, it is about a men's topwear.

The data itself is comprised of a train dataset, so we will define our training set as follows.

In [ ]:
dataset = ds['train']

Awesome, so now we hve seen what our dataset looks like, it is time to load the CLIP model and perform preprocessing.

## 3. Load CLIP Model and Preprocessing

The CLIP model version we will use is `ViT-B/32`. The model is moved to the appropriate device, a GPU if it is available, otherwise CPU.

In [ ]:
import clip
import torch

device            = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# OpenAI CLIP model and preprocessing
model, preprocess = clip.load("ViT-B/32", jit=False, device=device)

Let us take a look at how the pretrained CLIP model performs image classification on this dataset.

We will apply CLIP to classify three images from our dataset by comparing their visual features with textual descriptions of subcategories. It processes and normalizes the features of the images and subcategory texts, calculates their similarity, and predicts the subcategory for each image. Finally, it visualizes the images alongside their predicted and actual subcategories in a plot.

In [ ]:
import matplotlib.pyplot as plt

# Select indices for three images: in positions 0, 2, and 10
indices = [0, 2, 10]

# Get the list of possible subcategories from the dataset
subcategories = list(set(example['subCategory'] for example in dataset))

# Tokenize the text descriptions of each image under the form "a photo of <subcategory>" 
text_inputs = torch.cat([clip.tokenize(f"a photo of {c}") for c in subcategories]).to(device)

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loop through the indices and process each image
for i, idx in enumerate(indices):
    # Select an example image from the dataset
    example     = dataset[idx]
    image       = example['image']
    subcategory = example['subCategory']

    # Preprocess the image
    image_input = preprocess(image).unsqueeze(0).to(device)

    # Encode the image and text tokens with CLIP
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        text_features  = model.encode_text(text_inputs)

    # Normalize the obtained image and text features
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features  /= text_features.norm(dim=-1, keepdim=True)

    # Calculate cosine similarity between image and text features
    similarity      = (100.0 * image_features @ text_features.T).softmax(dim=-1)

    # Get the index of the most similar (image,text) pair
    values, indices = similarity[0].topk(1)

    # Display the most similar image-text pair
    axes[i].imshow(image)
    axes[i].set_title(f"Predicted: {subcategories[indices[0]]}, Actual: {subcategory}")
    axes[i].axis('off')

# Show the plot
plt.tight_layout()
plt.show()

As we can see for the three images, our base CLIP model does not perform very well. It only identifies one of the three images correctly. Let us set up the process for fine-tuning the CLIP model to improve these predictions.

In [ ]:
print(f'Dataset categories [{len(subcategories)}]:')
for cat in subcategories:
    print(f'\t{cat}')

## 4. Processing the Dataset

First, we must split the dataset into training and validation sets. This step is crucial because it allows us to evaluate the performance of our model on unseen data, ensuring that the model generalizes well to new, real-world data rather than just the data it was trained on.

We allocate 80% of the original dataset to train the model and the remaining 20% to the validation set.

In [ ]:
from torch.utils.data import random_split

# Split the dataset into training and validation sets

train_size                 = int(0.8 * len(dataset))
val_size                   = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

Next, we create a custom dataset class.

* **`__init__` method**: Initializes the dataset object with data and sets up a series of transformations to preprocess the images. The transformations include resizing the images to 224x224 pixels, converting them to tensors, and normalizing them with specific mean and standard deviation values.
* **`__len__` method**: Returns the number of samples in the dataset.
* **`__getitem__` method**: Retrieves an image and its corresponding subcategory from the dataset. The image is transformed using the predefined transformations, and the subcategory is converted to a label by finding its index in the subcategories list.

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset

# Define a custom dataset class
class FashionDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item        = self.data[idx]
        image       = item['image']
        subcategory = item['subCategory']
        label       = subcategories.index(subcategory)
        return self.transform(image), label

Next, we create the training and validation DataLoaders.

* `train_loader`: A DataLoader for training, with a batch size of 32 and shuffling enabled to randomize the order of samples.
* `val_loader`: A DataLoader for validation, with a batch size of 32 and shuffling disabled to maintain the order of samples.

In [ ]:
from torch.utils.data import DataLoader

# Create DataLoaderd for training and validation

train_loader = DataLoader(FashionDataset(train_dataset), batch_size=32, shuffle=True)
val_loader   = DataLoader(FashionDataset(val_dataset),   batch_size=32, shuffle=False)

Next, we modify the model to include a classifier for subcategories at the output.

* **`__init__` method**: Initializes the fine-tuning model with a base CLIP model and a new linear classifier for the subcategories. The linear layer has `num_classes` output units, corresponding to the number of subcategories.
* **`forward` method**: Passes the input images through the base CLIP model to extract features (without updating the base model's weights) and then through the new classifier to predict the subcategory.

In [ ]:
import torch.nn as nn

# Modify the model to include a classifier for subcategories
class CLIPFineTuner(nn.Module):
    def __init__(self, model, num_classes):
        super(CLIPFineTuner, self).__init__()
        self.model = model
        self.classifier = nn.Linear(model.visual.output_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            # Convert the model weights to float32
            features = self.model.encode_image(x).float()  
        return self.classifier(features)

Finally, we instantiate the fine-tuning model.

* `num_classes`: The number of unique subcategories in the dataset.
* `model_ft`: An instance of the `CLIPFineTuner` class, set up for fine-tuning on the subcategory classification task, and moved to the specified device (GPU or CPU).

In [ ]:
num_classes = len(subcategories)
model_ft    = CLIPFineTuner(model, num_classes).to(device)

Let us now define the loss function and optimizer.

### 5. Define Loss Function and Optimizer

* `criterion`: The loss function is Cross-Entropy, which is suitable for multi-class classification tasks.
* `optimizer`: The selected optimizer is Adam, applied only to the parameters of the classifier layer (`model_ft.classifier.parameters()`) with a learning rate of 0.0001.

In [ ]:
import torch.optim as optim

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_ft.classifier.parameters(), lr=1e-4)

### 6. Fine-Tuning the CLIP Model

Let us explain the training loop.

**Training:**

* `num_epochs`: Specifies the number of epochs.
* Training mode: The model is set to training mode using `model_ft.train()`.
* Progress bar: A progress bar (`tqdm`) is used to track the progress of the training loop, displaying the current epoch and loss.
* Training steps:
  - For each batch of images and labels obtained from the `train_loader`:
    - Move the images and labels to the specified device (CPU or GPU).
    - Zero the gradients using `optimizer.zero_grad()`.
    - Forward pass: Compute the model's outputs.
    - Compute the loss using `criterion`.
    - Backward pass: Compute the loss gradients using `loss.backward()`.
    - Update the model parameters using `optimizer.step()`.
    - Accumulate the epoch loss.
    - Update the progress bar description with the average loss for the epoch.
- After each epoch, the average loss for the epoch is printed out.

**Validation:**
- Evaluation mode: The model is set to evaluation mode using `model_ft.eval()`.
- Accuracy calculation:
  - Disable gradient computation with `torch.no_grad()`.
  - For each batch of images and labels obtained from `val_loader`:
    - Move the images and labels to the specified device.
    - Forward pass: Compute the model's outputs.
    - Get the predicted labels by finding the class with the highest score using `torch.max`.
    - Update the total number of labels and the count of correct predictions.
  - Calculate and print the validation accuracy as a percentage.

**Save the Fine-Tuned Model:**
The state dictionary of the fine-tuned model is saved to a file named `clip_finetuned.pth`.

In [ ]:
from tqdm import tqdm

loss_dict    = {}
val_acc_dict = {}

# Number of epochs for training
num_epochs = 10

# Training loop
for epoch in range(num_epochs):
    model_ft.train()  # Put the model into training mode
    running_loss = 0.0  # Initialize the loss of the current epoch
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}, Loss: 0.0000")  # Initialize progress bar

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)  # Move images and labels to the device (GPU or CPU)
        optimizer.zero_grad()       # Clear the gradients of all optimized variables
        outputs = model_ft(images)  # Forward pass: compute predicted outputs by passing inputs to the model
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Backward pass: compute gradient of the loss with respect to model parameters
        optimizer.step()  # Update model's parameters using the loss gradient

        running_loss += loss.item()  # Update the epoch's loss
        # Update the progress bar with current loss
        pbar.set_description(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")  

    # Print average loss for the epoch
    avg_loss = running_loss/len(train_loader)
    loss_dict[epoch+1] = avg_loss
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')  

    # Validation step
    model_ft.eval()        # Put the model in evaluation mode
    correct = 0            # Initialize correct predictions counter
    total   = 0            # Initialize total samples counter

    with torch.no_grad():  # Disable gradient calculation during validation
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)  # Move images and labels to the device
            outputs        = model_ft(images) # Forward pass: compute predicted outputs by passing inputs to the model
            _, predicted   = torch.max(outputs.data, 1)  # Get the class label with the highest probability
            total         += labels.size(0)              # Update total samples
            correct       += (predicted == labels).sum().item()  # Update correct predictions

    val_acc = 100 * correct / total
    val_acc_dict[epoch+1] = val_acc
    print(f'Validation Accuracy: {val_acc}%')  # Print validation accuracy for the epoch

# Save the fine-tuned model
torch.save(model_ft.state_dict(), 'clip_finetuned.pth')

In [ ]:
import matplotlib.pyplot as plt

# Loss epochs and values
epochs_loss = list(loss_dict.keys())
values_loss = list(loss_dict.values())

# Validation accuracy epochs and values
epochs_val_acc = list(val_acc_dict.keys())
values_val_acc = list(val_acc_dict.values())

# Create a pannel for two plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

# Plot the loss
ax1.plot(epochs_loss, values_loss, marker='o', linestyle='-', color='b')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training loss')
ax1.grid(True)

# Plot the validation accuracy
ax2.plot(epochs_val_acc, values_val_acc, marker='o', linestyle='-', color='g')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation accuracy (%)')
ax2.set_title('Validation accuracy')
ax2.grid(True)

# Adjust the layout e save the plot to a PNG file
plt.tight_layout()
plt.savefig('clip_finetuned_loss_acc.png') 
plt.show()


Each epoch takes around 3 minutes to run. Since we have 10 epochs, the fine-tunning takes roughly 30 minutes.

In the next screenshot we show the loss and accuracy obtained during the fine-tuning.

![Fine-tuning CLIP](../fig/clip_fintuning_02_training.png)

We observe that the fine-tuning process is successful, with the model showing significant improvements in both training loss and validation accuracy across the epochs. The final validation accuracy of 94.81% is a strong result, indicating that the model has effectively learned from the training data and is performing well on validation data. The gradual decrease in training loss and steady increase in validation accuracy reflect a well-conducted training process with no signs of overfitting or underfitting.

 Let us now take a look at how our new model performs on the same images we tested earlier.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision import transforms

# Load the saved model weights
model_ft.load_state_dict(torch.load('clip_finetuned.pth'))
model_ft.eval()  # Put the model into evaluation mode

# Define the indices for the three images we evaluated before
indices = [0, 2, 10]

# Transformations to apply to the images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
])

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loop through the indices and process each image
for i, idx in enumerate(indices):
    # Get the image and label from the dataset
    item       = dataset[idx]
    image      = item['image']
    true_label = item['subCategory']

    # Transform the image
    image_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Perform inference
    with torch.no_grad():
        output                 = model_ft(image_tensor)
        _, predicted_label_idx = torch.max(output, 1)
        predicted_label        = subcategories[predicted_label_idx.item()]

    # Display the image in the subplot
    axes[i].imshow(image)
    axes[i].set_title(f'True label: {true_label}\nPredicted label: {predicted_label}')
    axes[i].axis('off')

# Show the plot
plt.tight_layout()
plt.show()

Our fine-tuned CLIP model successfully predicts the labels for the three images.